In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Embedding models

The Embeddings class is a class designed for interfacing with text embedding models. There are lots of embedding model providers (OpenAI, Cohere, Hugging Face, etc) - this class is designed to provide a standard interface for all of them.

Embeddings create a vector representation of a piece of text. This is useful because it means we can think about text in the vector space, and do things like semantic search where we look for pieces of text that are most similar in the vector space.

The base Embeddings class in LangChain provides two methods: one for embedding documents and one for embedding a query. The former takes as input multiple texts, while the latter takes a single text. The reason for having these as two separate methods is that some embedding providers have different embedding methods for documents (to be searched over) vs queries (the search query itself).

In [ ]:
docs = [
    "cats eat and sleep",
    "dogs eat and bark",
    "cars drive fast",
    "vehicles include trucks and cars"
]

Embeddings create a vector representation of a piece of text. This is useful because it means we can think about text in the vector space, and do things like semantic search where we look for pieces of text that are most similar in the vector space.

The base Embeddings class in LangChain provides two methods: one for embedding documents and one for embedding a query. The former, `.embed_documents`, takes as input multiple texts, while the latter, `.embed_query`, takes a single text. The reason for having these as two separate methods is that some embedding providers have different embedding methods for documents (to be searched over) vs queries (the search query itself).

- `.embed_query`  will return a list of floats,
- `.embed_documents` returns a list of lists of floats.

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [ ]:
from pprint import pprint

## OpenAI's `text-embedding-3-small` Model Specifications

The **1536 dimensions** is the default output size for OpenAI's `text-embedding-3-small` model. This is a design choice made by OpenAI when they created this embedding model.

---

### Key Points About the 1536 Dimension Size

#### 1. Model Architecture Design
- OpenAI designed `text-embedding-3-small` to output **1536-dimensional vectors** by default.
- This size balances **performance** and **computational efficiency**.
- It is smaller than the larger `text-embedding-3-large` model, which outputs **3072 dimensions**.

#### 2. Why 1536 Specifically?
- **Powers of 2:** 1536 = 3 × 512, where 512 is 2⁹ (a common size in neural networks).
- **Computational Efficiency:** This size works well with modern GPU architectures.
- **Information Capacity:** 1536 dimensions provide enough capacity to capture semantic meaning effectively.


In [ ]:
from langchain_openai import OpenAIEmbeddings
# Notebook-friendly version with nicer progress bars
from tqdm.notebook import tqdm

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model="text-embedding-3-small")      

In [ ]:
embeddings = openai_embed_model.embed_documents(docs)

In [ ]:
print(f"The length of the embeddings is {len(embeddings)}")

In [ ]:
embeddings

In [ ]:
# Loop through each embedding vector in the embeddings list
for embedding in embeddings:
    # Print the length of the current embedding vector
    # This shows how many dimensions (features) each embedding has
    print(len(embedding))

#### Customizable Dimensions
You can actually change the output dimensions using OpenAI's embedding models:

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Default 1536 dimensions
openai_embed_default = OpenAIEmbeddings(model="text-embedding-3-small")

# Custom dimensions (can be reduced for efficiency)
openai_embed_custom = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=512  # Reduce to 512 dimensions
)

In [ ]:
embeddings_custom = openai_embed_custom.embed_documents(docs)
embeddings_custom

In [ ]:
# Loop through each embedding vector in the embeddings list
for embedding in embeddings_custom:
    # Print the length of the current embedding vector
    # This shows how many dimensions (features) each embedding has
    print(len(embedding))

Embedding Model Dimension Comparison:
- text-embedding-3-small: 1536 dimensions (default)
- text-embedding-3-large: 3072 dimensions
- text-embedding-ada-002: 1536 dimensions
- Sentence Transformers: Varies (e.g., 384, 768, 1024)

# Higher dimensions (1536+)
- ✅ Better semantic representation
- ✅ More nuanced understanding
- ❌ More storage space
- ❌ Slower similarity computations

# Lower dimensions (256-512)
- ✅ Faster computations
- ✅ Less storage needed
- ❌ Potential loss of semantic detail

In [ ]:
len(embeddings[0])

In [ ]:
docs

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix between all pairs of document embeddings.
# The resulting sim_matrix is a square matrix where each entry [i, j] represents
# the similarity between document i and document j. Values range from -1 (opposite)
# to 1 (identical), with 1s on the diagonal (each document is identical to itself).
sim_matrix = cosine_similarity(embeddings)
sim_matrix  # Shows how similar each document is to every other document.

## Open Source Embedding Models on HuggingFace

`langchain-huggingface` integrates seamlessly with LangChain, providing an efficient and effective way to utilize Hugging Face models within the LangChain ecosystem.

`HuggingFaceEmbeddings`uses `sentence-transformers` embeddings. It computes the embedding locally, using your computer resources and allows you to access open or open source embedding LLMs hosted on HuggingFace.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# check out model details here: https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1
model_name = "mixedbread-ai/mxbai-embed-large-v1"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
)

In [ ]:
embeddings = hf_embeddings.embed_documents(docs)

In [ ]:
len(embeddings)

In [ ]:
len(embeddings[0])

In [ ]:
docs

In [ ]:
sim_matrix = cosine_similarity(embeddings)
sim_matrix

## Build a small search engine!

### Load Knowledgebase documents

We have 10 Document in our Knowledge Base

In [ ]:
documents = [
    'Quantum mechanics describes the behavior of very small particles.',
    'Photosynthesis is the process by which green plants make food using sunlight.',
    "Shakespeare's plays are a testament to English literature.",
    'Artificial Intelligence aims to create machines that can think and learn.',
    'The pyramids of Egypt are historical monuments that have stood for thousands of years.',
    'Biology is the study of living organisms and their interactions with the environment.',
    'Music therapy can aid in the mental well-being of individuals.',
    'The Milky Way is just one of billions of galaxies in the universe.',
    'Economic theories help understand the distribution of resources in society.',
    'Yoga is an ancient practice that involves physical postures and meditation.'
]

In [ ]:
len(documents)

In [ ]:
### Get document embeddings
document_embeddings = openai_embed_model.embed_documents(documents)

In [ ]:
document_embeddings

### Let's try to find the most similar document for one query

In [ ]:
new_text = 'What is AI?'
new_text

In [ ]:
query_embedding = openai_embed_model.embed_query(new_text)

In [ ]:
len(query_embedding)

Query vs. Documents Similarity Analysis

- Comparing one query embedding: *"What is AI?"*
- Against 10 document embeddings from the knowledge base

- Objective: Find the document most similar to the query using cosine similarity

In [ ]:
# Compute the cosine similarity between the query embedding and each document embedding
cosine_similarities = cosine_similarity([query_embedding], document_embeddings)
cosine_similarities

In [ ]:
print(cosine_similarities)

In [ ]:
# From above we can see the document in 4th position is the most similar to the query "What is AI?"

most_similar_document = documents[3]
most_similar_document

We can also get is using `np.argmax`

In [ ]:
import numpy as np

documents[np.argmax(cosine_similarities[0])]    

### Create Search Engine function

In [ ]:
def semantic_search_engine(query, embedder_model):
  query_embedding = embedder_model.embed_query(query)
  cos_scores = cosine_similarity([query_embedding], document_embeddings)[0]
  top_result_id = np.argmax(cos_scores)
  return documents[top_result_id]

### Try out the function

In [ ]:
new_sentence = 'Tell me about AI'
semantic_search_engine(new_sentence, openai_embed_model)

In [ ]:
new_sentence = 'Do you know about the pyramids?'
semantic_search_engine(new_sentence, openai_embed_model)

In [ ]:
new_sentence = 'How do plants survive?'
semantic_search_engine(new_sentence, openai_embed_model)